In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_5.py ──
"""
Shared infrastructure for MLFP02 Exercise 5 — Linear Regression.

Contains: HDB resale data loading, feature engineering, OLS fitting utilities,
diagnostic helpers, and visualisation helpers. Technique-specific code (the
derivation walk-throughs, the WLS weight construction, the polynomial/dummy
matrices) lives in the per-technique files, not here.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ════════════════════════════════════════════════════════════════════════

NUMERIC_FEATURES: list[str] = [
    "floor_area_sqm",
    "storey_midpoint",
    "remaining_lease_years",
]
TARGET: str = "resale_price"
BASE_FLAT_TYPE: str = "3 ROOM"

OUTPUT_DIR = Path("outputs") / "ex5_linear_regression"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale flat transactions
# ════════════════════════════════════════════════════════════════════════


def load_hdb_clean() -> pl.DataFrame:
    """Load HDB resale data, engineer numeric features, drop nulls.

    Returns a polars DataFrame with columns:
      - floor_area_sqm (Float)
      - storey_midpoint (Float, midpoint of '07 TO 09' range)
      - remaining_lease_years (Float, 99 - years_elapsed_since_commence)
      - resale_price (target, SGD)
      - flat_type (categorical)
      - town (categorical)
    """
    loader = MLFPDataLoader()
    hdb = loader.load("mlfp01", "hdb_resale.parquet")

    hdb = hdb.with_columns(
        pl.col("month").str.to_date("%Y-%m").alias("transaction_date"),
    )
    hdb_recent = hdb.filter(pl.col("transaction_date") >= pl.date(2020, 1, 1))

    hdb_recent = hdb_recent.with_columns(
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2
        ).alias("storey_midpoint"),
        (99 - (pl.col("transaction_date").dt.year() - pl.col("lease_commence_date")))
        .cast(pl.Float64)
        .alias("remaining_lease_years"),
    )

    return hdb_recent.drop_nulls(
        subset=[*NUMERIC_FEATURES, TARGET],
    )


def build_design_matrix(
    df: pl.DataFrame, features: list[str] | None = None
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Build (X_with_intercept, y, feature_names) from a cleaned HDB frame."""
    features = features or NUMERIC_FEATURES
    y = df[TARGET].to_numpy().astype(np.float64)
    X_raw = df.select(features).to_numpy().astype(np.float64)
    n_obs = X_raw.shape[0]
    X = np.column_stack([np.ones(n_obs), X_raw])
    feature_names = ["intercept"] + list(features)
    return X, y, feature_names


# ════════════════════════════════════════════════════════════════════════
# OLS CORE — the workhorse every technique reuses
# ════════════════════════════════════════════════════════════════════════


def fit_ols(X: np.ndarray, y: np.ndarray) -> dict[str, Any]:
    """Fit OLS via the normal equation and return core statistics.

    Returns a dict with keys: beta, y_hat, residuals, XtX_inv, SSR, SST, SSE,
    R2, adj_R2, sigma_hat, se_beta, t_stats, p_values, f_stat, f_p_value, n, k.
    """
    n, k = X.shape
    XtX = X.T @ X
    XtX_inv = np.linalg.inv(XtX)
    beta = XtX_inv @ X.T @ y

    y_hat = X @ beta
    residuals = y - y_hat

    SSR = float(np.sum(residuals**2))
    SST = float(np.sum((y - y.mean()) ** 2))
    SSE = float(np.sum((y_hat - y.mean()) ** 2))

    sigma_sq = SSR / (n - k)
    sigma_hat = float(np.sqrt(sigma_sq))
    se_beta = np.sqrt(sigma_sq * np.diag(XtX_inv))
    t_stats = beta / se_beta
    # sf (survival function) keeps precision in the far tail; 1 - cdf
    # rounds to exactly 0 once the tail drops below ~1e-16.
    p_values = 2 * stats.t.sf(np.abs(t_stats), df=n - k)

    r2 = 1 - SSR / SST
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k)
    f_stat = (SSE / (k - 1)) / (SSR / (n - k))
    f_p = stats.f.sf(f_stat, dfn=k - 1, dfd=n - k)

    return {
        "beta": beta,
        "y_hat": y_hat,
        "residuals": residuals,
        "XtX_inv": XtX_inv,
        "SSR": SSR,
        "SST": SST,
        "SSE": SSE,
        "R2": float(r2),
        "adj_R2": float(adj_r2),
        "sigma_hat": sigma_hat,
        "se_beta": se_beta,
        "t_stats": t_stats,
        "p_values": p_values,
        "f_stat": float(f_stat),
        "f_p_value": float(f_p),
        "n": int(n),
        "k": int(k),
    }


def format_p_value(p: float) -> str:
    """Render a p-value for printing, including the underflow case.

    Returns e.g. "= 3.20e-05", or "< 1e-300" when the tail probability is
    smaller than the smallest representable double and evaluates to 0.
    """
    return f"= {p:.2e}" if p > 0 else "< 1e-300"


def print_coef_table(names: list[str], fit: dict[str, Any]) -> None:
    """Print coefficient / SE / t / p table for an OLS fit."""
    beta = fit["beta"]
    se = fit["se_beta"]
    t = fit["t_stats"]
    p = fit["p_values"]
    print(f"\n{'Feature':<25} {'β':>14} {'SE(β)':>12} {'t':>8} {'p':>10} {'Sig':>4}")
    print("-" * 78)
    for i, name in enumerate(names):
        if p[i] < 0.001:
            sig = "***"
        elif p[i] < 0.01:
            sig = "**"
        elif p[i] < 0.05:
            sig = "*"
        else:
            sig = "ns"
        print(
            f"{name:<25} {beta[i]:>14,.2f} {se[i]:>12,.2f} "
            f"{t[i]:>8.2f} {(f'{p[i]:.2e}' if p[i] > 0 else '<1e-300'):>10} {sig:>4}"
        )


# ════════════════════════════════════════════════════════════════════════
# DIAGNOSTICS — VIF, Breusch-Pagan, residual shape
# ════════════════════════════════════════════════════════════════════════


def compute_vif(X_raw: np.ndarray, feature_names: list[str]) -> dict[str, float]:
    """Variance Inflation Factor for each feature (no intercept column)."""
    n = X_raw.shape[0]
    results: dict[str, float] = {}
    for j in range(X_raw.shape[1]):
        other = [i for i in range(X_raw.shape[1]) if i != j]
        Xo = np.column_stack([np.ones(n), X_raw[:, other]])
        yj = X_raw[:, j]
        beta_j = np.linalg.lstsq(Xo, yj, rcond=None)[0]
        yhat_j = Xo @ beta_j
        ss_res = np.sum((yj - yhat_j) ** 2)
        ss_tot = np.sum((yj - yj.mean()) ** 2)
        r2_j = 1 - ss_res / ss_tot
        results[feature_names[j]] = (
            float(1.0 / (1.0 - r2_j)) if r2_j < 1 else float("inf")
        )
    return results


def breusch_pagan(residuals: np.ndarray, X_raw: np.ndarray) -> tuple[float, float]:
    """Breusch-Pagan test for heteroscedasticity. Returns (BP statistic, p-value)."""
    n = X_raw.shape[0]
    e_sq = residuals**2
    Xbp = np.column_stack([np.ones(n), X_raw])
    beta = np.linalg.lstsq(Xbp, e_sq, rcond=None)[0]
    pred = Xbp @ beta
    sse = np.sum((e_sq - pred) ** 2)
    sst = np.sum((e_sq - e_sq.mean()) ** 2)
    r2 = 1 - sse / sst
    bp_stat = n * r2
    p = stats.chi2.sf(bp_stat, df=X_raw.shape[1])
    return float(bp_stat), float(p)


# ════════════════════════════════════════════════════════════════════════
# VISUALISE — residual plots + actual-vs-predicted
# ════════════════════════════════════════════════════════════════════════


def save_residual_diagnostics(
    y_hat: np.ndarray,
    residuals: np.ndarray,
    feature_col: np.ndarray,
    feature_label: str,
    filename: str,
) -> Path:
    """Four-panel diagnostic figure: residuals vs fitted, histogram, Q-Q, vs feature."""
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[
            "Residuals vs Fitted",
            "Residual Histogram",
            "Q-Q Plot",
            f"Residuals vs {feature_label}",
        ],
    )
    sample = min(3000, len(residuals))
    fig.add_trace(
        go.Scatter(
            x=y_hat[:sample],
            y=residuals[:sample],
            mode="markers",
            marker={"size": 2, "opacity": 0.3},
            name="Residuals",
        ),
        row=1,
        col=1,
    )
    fig.add_hline(y=0, row=1, col=1, line_dash="dash")
    fig.add_trace(go.Histogram(x=residuals, nbinsx=50, name="Residuals"), row=1, col=2)
    sorted_resid = np.sort(residuals)
    step = max(1, len(sorted_resid) // 2000)
    theoretical = stats.norm.ppf(np.linspace(0.001, 0.999, len(sorted_resid)))
    fig.add_trace(
        go.Scatter(
            x=theoretical[::step],
            y=sorted_resid[::step],
            mode="markers",
            marker={"size": 2},
            name="Q-Q",
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=feature_col[:sample],
            y=residuals[:sample],
            mode="markers",
            marker={"size": 2, "opacity": 0.3},
            name=f"vs {feature_label}",
        ),
        row=2,
        col=2,
    )
    fig.update_layout(height=600, title="Residual Diagnostics", showlegend=False)
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


def save_actual_vs_predicted(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    title: str,
    filename: str,
) -> Path:
    """Actual-vs-predicted scatter with the perfect-prediction diagonal."""
    sample = min(2000, len(y_true))
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=y_true[:sample],
            y=y_pred[:sample],
            mode="markers",
            marker={"size": 3, "opacity": 0.4},
            name="Predictions",
        )
    )
    lo, hi = float(y_true.min()), float(y_true.max())
    fig.add_trace(
        go.Scatter(
            x=[lo, hi],
            y=[lo, hi],
            mode="lines",
            line={"dash": "dash", "color": "red"},
            name="Perfect",
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="Actual Price (SGD)",
        yaxis_title="Predicted Price (SGD)",
        height=500,
    )
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 5.3: Weighted Least Squares (WLS)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Implement WLS when heteroscedasticity is present
#   - Estimate variance weights from fitted values
#   - Compare OLS and WLS coefficients and standard errors
#   - Understand when WLS improves inference vs point estimates
#
# PREREQUISITES: Exercise 5.1-5.2 (OLS, diagnostics, Breusch-Pagan)
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Load data and fit baseline OLS
#   2. Estimate variance function from residuals
#   3. Implement WLS: beta = (X'WX)^{-1}X'Wy
#   4. Compare OLS and WLS results
#   5. Visualise the effect of weighting
#
# THEORY:
#   When Var(e_i) = sigma_i^2 (not constant), OLS gives unbiased but
#   inefficient estimates. WLS weights each observation by 1/sigma_i^2,
#   giving less weight to high-variance observations.
#   beta_wls = (X'WX)^{-1}X'Wy where W = diag(1/sigma_i^2)
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots



## TASK 1 — Load Data and Fit Baseline OLS


In [ ]:
print("=" * 70)
print("  MLFP02 Exercise 5.3: Weighted Least Squares")
print("=" * 70)

hdb_clean = load_hdb_clean()
X, y, feature_names = build_design_matrix(hdb_clean)
n_obs, k = X.shape

fit_baseline = fit_ols(X, y)
beta_ols = fit_baseline["beta"]
residuals = fit_baseline["residuals"]
y_hat = fit_baseline["y_hat"]
r_squared = fit_baseline["R2"]
SST = fit_baseline["SST"]
SSR = fit_baseline["SSR"]

print(
    f"\n  Baseline OLS: R-squared={r_squared:.6f}, RMSE=${fit_baseline['sigma_hat']:,.0f}"
)



## THEORY — Why Weight Observations?


In [ ]:
# OLS treats every observation equally. But in housing data, a $1.2M
# executive flat has more price variation than a $300K 3-room flat —
# the "noise" is louder for expensive properties.
#
# Analogy: Imagine averaging exam scores from two classes. Class A has
# 30 students with tightly clustered scores (low variance). Class B
# has 30 students with wildly spread scores (high variance). A simple
# average weights them equally, but you would trust Class A's average
# more. WLS does exactly this — it trusts low-variance observations
# more than high-variance ones.
#
# WHY THIS MATTERS: A real estate analytics firm building an automated
# valuation model (AVM) for mortgage approvals needs honest uncertainty
# for EACH flat. OLS assumes one noise level for every flat, so its
# prediction interval is the same width for a cheap 3-room and an
# expensive executive flat — too narrow for the noisy, expensive end
# and too wide for the quiet, cheap end. WLS models the noise level, so
# the interval can widen where prices are genuinely more variable.



## TASK 2 — Estimate Variance Function


In [ ]:
# We model variance as a function of X: Var(e_i) ~ (X @ gamma)^2
# Use |residuals| as a proxy for standard deviation.

print(f"\n=== Estimating Variance Function ===")

abs_resid = np.abs(residuals)

# TODO: Fit |residuals| ~ X to get a variance model
# Hint: use np.linalg.lstsq(X, abs_resid, rcond=None)[0]
w_beta = ____

# TODO: Compute estimated variance per observation
# Hint: variance_hat = np.maximum((X @ w_beta) ** 2, 1e-6)
variance_hat = ____

# TODO: Compute weights = 1 / variance_hat
weights = ____

print(f"  Weight range: [{weights.min():.3e}, {weights.max():.3e}]")
print(f"  Weight ratio (max/min): {weights.max() / weights.min():.1f}x")
print(
    f"  Observations with above-median weight: {np.sum(weights > np.median(weights)):,}"
)



## TASK 3 — Implement WLS: beta = (X'WX)^{-1}X'Wy


In [ ]:
print(f"\n=== Weighted Least Squares ===")

# Never build W = diag(weights): with n = 24,904 that dense n x n matrix
# is ~5 GB. Scaling each row of X by its weight gives the same X'W.
# TODO: Row-scale X by the weights (X'W without the n x n matrix)
# Hint: broadcast weights as a column — weights[:, None]
Xw = ____

# TODO: Compute X'WX and X'Wy from the row-scaled matrix
# Hint: Xw.T @ X and Xw.T @ y
XtWX = ____
XtWy = ____

# TODO: Solve for WLS coefficients
# Hint: np.linalg.solve(XtWX, XtWy)
beta_wls = ____

y_hat_wls = X @ beta_wls
residuals_wls = y - y_hat_wls
ssr_wls = float(np.sum(residuals_wls**2))
r2_wls = 1 - ssr_wls / SST

# WLS standard errors
sigma_sq_wls = float(np.sum(weights * residuals_wls**2)) / (n_obs - k)
XtWX_inv = np.linalg.inv(XtWX)
se_wls = np.sqrt(sigma_sq_wls * np.diag(XtWX_inv))



### Checkpoint 3


In [ ]:
assert len(beta_wls) == k, "WLS must have same number of coefficients"
assert ssr_wls > 0, "WLS SSR must be positive"
print("--- Checkpoint 3 passed --- WLS fitted\n")



## TASK 4 — Compare OLS and WLS


In [ ]:
print(f"{'Feature':<25} {'OLS beta':>12} {'WLS beta':>12} {'Delta':>10}")
print("-" * 62)
for i, name in enumerate(feature_names):
    delta = beta_wls[i] - beta_ols[i]
    print(f"{name:<25} {beta_ols[i]:>12,.2f} {beta_wls[i]:>12,.2f} {delta:>+10,.2f}")

print(f"\nOLS R-squared  = {r_squared:.6f}")
print(f"WLS R-squared  = {r2_wls:.6f}")
print(f"OLS RMSE = ${np.sqrt(SSR / n_obs):,.0f}")
print(f"WLS RMSE = ${np.sqrt(ssr_wls / n_obs):,.0f}")

# Standard error comparison
print(f"\n{'Feature':<25} {'OLS SE':>12} {'WLS SE':>12}")
print("-" * 52)
for i, name in enumerate(feature_names):
    print(f"{name:<25} {fit_baseline['se_beta'][i]:>12,.2f} {se_wls[i]:>12,.2f}")

# INTERPRETATION: WLS coefficients may differ from OLS when
# heteroscedasticity is present. WLS gives more reliable standard
# errors and confidence intervals. If OLS and WLS coefficients are
# similar, the heteroscedasticity doesn't much affect the point
# estimates — but the SEs are still more trustworthy from WLS.



### Checkpoint 4


In [ ]:
assert all(se > 0 for se in se_wls), "All WLS standard errors must be positive"
print("\n--- Checkpoint 4 passed --- OLS vs WLS compared\n")



## VISUALISE — WLS Actual vs Predicted


In [ ]:
path = save_actual_vs_predicted(
    y,
    y_hat_wls,
    title="WLS: Actual vs Predicted Price",
    filename="03_wls_actual_vs_predicted.html",
)
print(f"Saved: {path}")

# --- Residual variance: OLS vs WLS (before/after weighting) ---
sample = min(3000, n_obs)
fig_var = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "OLS Residuals vs Fitted",
        "WLS Residuals vs Fitted",
    ],
)
fig_var.add_trace(
    go.Scatter(
        x=y_hat[:sample].tolist(),
        y=residuals[:sample].tolist(),
        mode="markers",
        marker={"size": 2, "opacity": 0.3, "color": "steelblue"},
        name="OLS",
    ),
    row=1,
    col=1,
)
fig_var.add_trace(
    go.Scatter(
        x=y_hat_wls[:sample].tolist(),
        y=residuals_wls[:sample].tolist(),
        mode="markers",
        marker={"size": 2, "opacity": 0.3, "color": "#D97706"},
        name="WLS",
    ),
    row=1,
    col=2,
)
# TODO: Draw a zero-residual reference line on BOTH panels.
# Hint: fig_var.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=...)
____
____
fig_var.update_layout(
    title="Residual Spread: OLS vs WLS — Does Weighting Reduce the Fan Shape?",
    height=400,
    width=900,
    showlegend=False,
)
fig_var.update_xaxes(title_text="Predicted ($)", row=1, col=1)
fig_var.update_xaxes(title_text="Predicted ($)", row=1, col=2)
fig_var.update_yaxes(title_text="Residual ($)", row=1, col=1)
fig_var.update_yaxes(title_text="Residual ($)", row=1, col=2)
path_var = OUTPUT_DIR / "03_wls_residual_comparison.html"
fig_var.write_html(str(path_var))
print(f"Saved: {path_var}")

# --- Predicted-vs-actual comparison: OLS vs WLS side by side ---
fig_cmp = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        f"OLS (R-sq={r_squared:.4f})",
        f"WLS (R-sq={r2_wls:.4f})",
    ],
)
lo, hi = float(y.min()), float(y.max())
for col_idx, (pred, label) in enumerate([(y_hat, "OLS"), (y_hat_wls, "WLS")], start=1):
    fig_cmp.add_trace(
        go.Scatter(
            x=y[:sample].tolist(),
            y=pred[:sample].tolist(),
            mode="markers",
            marker={"size": 2, "opacity": 0.3},
            name=label,
        ),
        row=1,
        col=col_idx,
    )
    fig_cmp.add_trace(
        go.Scatter(
            x=[lo, hi],
            y=[lo, hi],
            mode="lines",
            line={"dash": "dash", "color": "red"},
            showlegend=False,
        ),
        row=1,
        col=col_idx,
    )
fig_cmp.update_layout(
    title="Actual vs Predicted: OLS vs WLS — How Does Weighting Change Predictions?",
    height=400,
    width=900,
)
fig_cmp.update_xaxes(title_text="Actual ($)", row=1, col=1)
fig_cmp.update_xaxes(title_text="Actual ($)", row=1, col=2)
fig_cmp.update_yaxes(title_text="Predicted ($)", row=1, col=1)
fig_cmp.update_yaxes(title_text="Predicted ($)", row=1, col=2)
path_cmp = OUTPUT_DIR / "03_wls_ols_comparison.html"
fig_cmp.write_html(str(path_cmp))
print(f"Saved: {path_cmp}")



## APPLY — Automated Valuation Model (AVM) for Mortgage Approval


In [ ]:
# SCENARIO: A Singapore proptech firm builds an automated valuation
# model (AVM) for mortgage lenders. The AVM must state not just a point
# estimate but how far THIS flat's price could plausibly fall from it.
#
# Two different intervals answer two different questions:
#   - CONFIDENCE interval for the MEAN price of flats like this one:
#       se_mean = sqrt(sigma^2 * x'(X'X)^{-1}x)   — shrinks with n
#   - PREDICTION interval for ONE flat's price:
#       se_pred = sqrt(sigma^2 * (1 + x'(X'X)^{-1}x)) — never shrinks
#     below the noise sigma, however much data you have.
# A valuation is about one flat, so it needs the PREDICTION interval.
# Under WLS the "1" becomes that flat's own estimated noise variance,
# so the interval is wider for noisier (larger, pricier) flats.
#
# BUSINESS IMPACT: With a 75% loan-to-value cap, the loan size moves
# 1:1 with the valuation. Quoting the narrow mean-CI as if it were
# the uncertainty of one flat understates the lender's valuation risk
# by more than an order of magnitude (compare the columns below).

print(f"\n--- Business Application: AVM Intervals for One Flat ---")
example_flats = {
    "~3-room (67 sqm, floor 8, 74y lease)": np.array([1.0, 67.0, 8.0, 74.0]),
    "~4-room (92 sqm, floor 8, 75y lease)": np.array([1.0, 92.0, 8.0, 75.0]),
    "~Executive (145 sqm, floor 8, 73y lease)": np.array([1.0, 145.0, 8.0, 73.0]),
}
sigma2_ols = fit_baseline["sigma_hat"] ** 2
print(
    f"  {'Flat':<42} {'Model':>5} {'Point':>12} "
    f"{'+/- mean CI':>12} {'+/- pred. PI':>13}"
)
pi_halfwidths = {}
for label, x_new in example_flats.items():
    leverage_ols = float(x_new @ fit_baseline["XtX_inv"] @ x_new)
    ci_ols = 1.96 * np.sqrt(sigma2_ols * leverage_ols)
    # TODO: OLS 95% PREDICTION half-width for one flat.
    # Hint: like ci_ols, but add the flat's own noise: sigma^2 * (1 + leverage)
    pi_ols = ____

    # WLS: this flat's own noise variance comes from the variance model
    var_new = max(float(x_new @ w_beta) ** 2, 1e-6)
    leverage_wls = float(x_new @ XtWX_inv @ x_new)
    ci_wls = 1.96 * np.sqrt(sigma_sq_wls * leverage_wls)
    # TODO: WLS 95% PREDICTION half-width — the flat's own variance replaces the "1".
    # Hint: same shape as ci_wls, with var_new added inside the square root
    pi_wls = ____
    pi_halfwidths[label] = (pi_ols, pi_wls)

    print(f"  {label:<42} {'OLS':>5} ${x_new @ beta_ols:>11,.0f} ${ci_ols:>11,.0f} ${pi_ols:>12,.0f}")
    print(f"  {'':<42} {'WLS':>5} ${x_new @ beta_wls:>11,.0f} ${ci_wls:>11,.0f} ${pi_wls:>12,.0f}")

ols_pis = [v[0] for v in pi_halfwidths.values()]
wls_pis = [v[1] for v in pi_halfwidths.values()]
print(
    f"\n  OLS prediction intervals span ${min(ols_pis):,.0f}-${max(ols_pis):,.0f}: "
    f"essentially one width for every flat."
)
print(
    f"  WLS prediction intervals span ${min(wls_pis):,.0f}-${max(wls_pis):,.0f}: "
    f"they track each flat's estimated noise level."
)



### Checkpoint 5


In [ ]:
for pi_ols, pi_wls in pi_halfwidths.values():
    assert pi_ols > 0 and pi_wls > 0, "Prediction intervals must be positive"
assert all(
    pi > 10 * 1.96 * np.sqrt(sigma2_ols * float(x @ fit_baseline["XtX_inv"] @ x))
    for pi, x in zip(ols_pis, example_flats.values())
), "A prediction interval for one flat must be far wider than the CI of the mean"
print("\n--- Checkpoint 5 passed --- confidence vs prediction intervals\n")



## REFLECTION


- Variance function estimation from OLS residuals
  - WLS implementation: beta = (X'WX)^{-1}X'Wy
  - Comparing OLS and WLS: coefficients, SEs, and R-squared
  - When WLS matters: heteroscedastic data with varying noise levels
  - Confidence interval (mean) vs prediction interval (one flat) for valuations

  NEXT: In 04_model_enrichment.py you'll extend the model with
  polynomial terms, interaction effects, dummy variables, and
  train/test evaluation.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED (5.3)")
print("=" * 70)
print(
)

print("--- Exercise 5.3 complete --- Weighted Least Squares")

